This has been run :)

but I noticed a bug with the draw tirgger count???

TODO: Investigate and fix that?

Imports

In [1]:
import db_operations
import helpers

import pandas as pd
from pymongo import UpdateOne

Constants

In [2]:
STAND_HEALS = [
    "Stealth Fiend, Amaviera",
    "Incorruptible Holy Light, Eufha",
    "Lady Healer of the Creaking World",
    "Invigorate Sage",
    "Zypsophila Fairy, Asher",
    "Two Of Us, Rhyme",
]
CRIT_HEALS = [
    'Cure Flare Dracokid',
    'Brilliant Floral, Uania',
    'Whistling Arrow of Recursion, Obifold',
    'Heartiness Tear Sorceress',
    'Alchemic Hedgehog',
    'Two Of Us, Flow',
]

DRAWS = [
    'Flare Veil Dragon',
    'Rouse Wildmaster, Riley',
    'Ameliorate Connector',
    'Protection Magic, Prorobi',
    'Serene Maiden, Lena',
    'Transparent Snowy Night, Beretoi',
]

REGALIS = [
    "Primordial Heartblaze",
    "Shroud in Darkness",
    "Sound of Wave",
    "Fire Regalis",
    "Union the Sky",
    "Protection: Twincast",
    "Radiance Caliburn",
    "Bracing Angel Ladder",
    "Forbidoll Surrogate",  "'Mysterious Joker' Joker",
    "Evergreen Transhpere", "'Homeroom-Ninja Mr. Shinobu' Mr. Shinobu",
    "Gratias Gradale",      "'Croket!' Croket",
]
REGALIS_MAP = {
    "Primordial Heartblaze": None,
    "Shroud in Darkness": None,
    "Sound of Wave": None,
    "Fire Regalis": None,
    "Union the Sky": None,
    "Protection: Twincast": None,
    "Radiance Caliburn": None,
    "Bracing Angel Ladder": None,
    "Forbidoll Surrogate": None,  
    "'Mysterious Joker' Joker": "Forbidoll Surrogate",
    "Evergreen Transhpere": None, 
    "'Homeroom-Ninja Mr. Shinobu' Mr. Shinobu": "Evergreen Transhpere",
    "Gratias Gradale": None,      
    "'Croket!' Croket": "Gratias Gradale",
}

Helper Functions

Get Data

In [14]:
full_df = db_operations.get_table('main_table', 'all_events')

Feature Engineering

In [ ]:
full_df.loc[:,'stand_heal_count'] = full_df.deck.apply(lambda deck: helpers.card_type_in_deck(deck, STAND_HEALS))

full_df.loc[:,'crit_heal_count']  = full_df.deck.apply(lambda deck: helpers.card_type_in_deck(deck, CRIT_HEALS))

full_df.loc[:,'draw_count']       = full_df.deck.apply(lambda deck: helpers.card_type_in_deck(deck, REGALIS))

full_df.loc[:, 'regalis_piece']   = full_df.deck.apply(lambda deck: helpers.extract_regalis(deck, REGALIS))

full_df.loc[:, 'boss'] = full_df.boss.apply(lambda boss: helpers.extract_card_name(boss))

# full_df.sort_values('rank', inplace=True)
full_df.head()

,_id,rank,name,boss,wins,nation,decklog,deck,location,region,date,regalis_piece,stand_heal_count,crit_heal_count,draw_count
0,69b018213efa47b42b6c188e,5,Ryuta,"Rewrite the Star, Vyrgilla",8,Dragon Empire,57S3E,{'RideDeck': {'DZ-SS04/001EN-R : Rewrite the S...,Rosemont,NA,"October 4, 2025",Protection: Twincast,0,0,1
1,69b018213efa47b42b6c188f,6,BigDaddyBrie,"Sugary and Scary Land, Heartluru",8,Dark States,741C4,{'RideDeck': {'DZ-BT09/SR08EN : Sugary and Sca...,Rosemont,NA,"October 4, 2025",Union the Sky,1,0,1
2,69b018213efa47b42b6c1890,7,Saaram,"Sealed Blaze of Arbitration, Bavsargra Aksayya",8,Dragon Empire,5JGCY,{'RideDeck': {'DZ-BT07/001EN : Sealed Blaze of...,Rosemont,NA,"October 4, 2025",Fire Regalis,4,0,1
3,69b018213efa47b42b6c1891,8,TKS|RedX,"Rewrite the Star, Vyrgilla",8,Dragon Empire,7T6BW,{'RideDeck': {'DZ-SS04/001EN-R : Rewrite the S...,Rosemont,NA,"October 4, 2025",Union the Sky,2,1,1
4,69b018213efa47b42b6c1892,9,Josh.,"Rewrite the Star, Vyrgilla",8,Dragon Empire,6Q7SL,{'RideDeck': {'DZ-SS04/001EN-R : Rewrite the S...,Rosemont,NA,"October 4, 2025",Union the Sky,3,0,1


Save

In [17]:

updates = []
for i, row in full_df.iterrows():
    filter = {'_id': row['_id']}
    update = {
        '$set': {
            'boss': row['boss'],
            'stand_heal_count': row['stand_heal_count'],
            'crit_heal_count': row['crit_heal_count'],
            'draw_count': row['draw_count'],
            'regalis_piece': row['regalis_piece'],
        }
    }
    updates.append(UpdateOne(filter, update))

db_operations.update_table('main_table', 'all_events',  updates)